In [ ]:
import pandas as pd
validation_data= pd.read_csv("type_classification-validation.csv")

In [3]:
# pip install azure-ai-inference
import os
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv
load_dotenv()

api_key = os.getenv("AZURE_DEEPSEK_API_KEY_1", "")
api_url = os.getenv("AZURE_DEEPSEEK_ENDPOINT_URL_1", "")
if not api_key:
  raise Exception("A key should be provided to invoke the endpoint")

client = ChatCompletionsClient(
    endpoint=api_url,
    credential=AzureKeyCredential(api_key)
)


In [ ]:
from tqdm import tqdm 
new_validation_df = pd.DataFrame(columns=["Sentence", "Result"])

for index, row in tqdm(validation_data.iterrows(), total=len(validation_data)):
  payload = {
    "messages": [
      {
        "role": "user",
        "content": "You are an AI Model that help to classify whether a sentence can be used to support information to create Class Diagram or Use Case Diagram or Activity Diagram\n\nBelow are given 5 sentences and whether it is useful for any of the diagram\n1. \"AP : As a case progresses , I need to record all the individuals and organizations that Verdict: take part in the case activities and the specific role they play .\"\nUseful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n2. \"Unless you are a celebrity or a good friend of Romano you will need a reservation .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n3. \"Therefore , there can be overlapping table reservations .\"\nVerdict: Useful for Class Diagram, Not Useful for Use Case Diagram, Not Useful for Activity Diagram\n\n4. \"These samples are sometimes sub - divided and distributed to multiple research teams or labs for different specialized observations .\"\nVerdict: Not Useful for Class Diagram, Useful for Use Case Diagram, Useful for Activity Diagram\n\n5. \"When the reservation party arrives at Romano 's the reservation is assigned to one waiter .\"\nVerdict: Useful for Class Diagram, Useful for Use Case Diagram, Not Useful for Activity Diagram\n\nUser will put a sentence and decide whether it will be useful for any of the category as an example output like this [Useful Class, Useful Use Case, Not Useful Activity]\n\n"
      },
        {
        "role": "user",
        "content": row['sentence']
      },
    ],
    "max_tokens": 2048
  }
  try:
    response = client.complete(payload)
    result = response.choices[0].message.content
    new_validation_df = new_validation_df.append({"Sentence": row['sentence'], "Result": result}, ignore_index=True)
  except Exception as e:
      print(e)
      print("Error at index: ", index, " Sentence: ", row['sentence'])
      new_validation_df = new_validation_df.append({"Sentence": row['sentence'], "Result": ""}, ignore_index=True)

new_validation_df.to_csv("DeepSeekR1-label-validation-5 example.csv")



In [1]:
import pandas as pd
r1_validated = pd.read_csv("DeepSeekR1-label-validation-5 example.csv")
r1_validated.columns = ["index", "Sentence", "Result"]
r1_validated["Extracted"] = r1_validated["Result"].str.extract(r"\[([^\]]+)\]")
r1_validated.to_csv("DeepSeekR1-label-validation-5 example.csv")